In [1]:
def smith_waterman(seq1, seq2, match=2, mismatch=-1, gap=-1):
    """
    Performs Smith-Waterman local sequence alignment on two strings.

    Parameters:
        seq1 (str): First biological sequence (represented on rows).
        seq2 (str): Second biological sequence (represented on columns).
        match (int): Score assigned for matching characters.
        mismatch (int): Penalty score for mismatching characters.
        gap (int): Penalty score for introducing a gap.

    Returns:
        tuple: (max_score, aligned_seq1, aligned_seq2)
    """
    m, n = len(seq1), len(seq2)

    # 1. Initialize the scoring matrix with zeros
    # Dimensions are (m + 1) x (n + 1) to account for initial gap conditions
    matrix = [[0] * (n + 1) for _ in range(m + 1)]

    max_score = 0
    max_pos = (0, 0)

    # 2. Fill the scoring matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # Calculate score for a diagonal move (match/mismatch)
            if seq1[i - 1] == seq2[j - 1]:
                diagonal_score = matrix[i - 1][j - 1] + match
            else:
                diagonal_score = matrix[i - 1][j - 1] + mismatch

            # Calculate scores for structural gaps (moving down or right)
            gap_seq1 = matrix[i - 1][j] + gap  # Deletion in seq2 / Gap in seq2
            gap_seq2 = matrix[i][j - 1] + gap  # Insertion in seq2 / Gap in seq1

            # Smith-Waterman critical step: No negative values allowed (lower-bounded by 0)
            matrix[i][j] = max(0, diagonal_score, gap_seq1, gap_seq2)

            # Keep track of the highest score tracking position
            if matrix[i][j] >= max_score:
                max_score = matrix[i][j]
                max_pos = (i, j)

    # 3. Traceback phase
    aligned1, aligned2 = [], []
    i, j = max_pos

    # Traceback continues until encountering a cell with a score of 0
    while i > 0 and j > 0 and matrix[i][j] > 0:
        current_score = matrix[i][j]

        # Determine whether a diagonal, up, or left step produced the current score
        if seq1[i - 1] == seq2[j - 1]:
            expected_diagonal = matrix[i - 1][j - 1] + match
        else:
            expected_diagonal = matrix[i - 1][j - 1] + mismatch

        if current_score == expected_diagonal:
            aligned1.append(seq1[i - 1])
            aligned2.append(seq2[j - 1])
            i -= 1
            j -= 1
        elif current_score == matrix[i - 1][j] + gap:
            aligned1.append(seq1[i - 1])
            aligned2.append('-')
            i -= 1
        else:
            aligned1.append('-')
            aligned2.append(seq2[j - 1])
            j -= 1

    # Reverse alignment strings since they were built backwards during traceback
    aligned_seq1 = ''.join(reversed(aligned1))
    aligned_seq2 = ''.join(reversed(aligned2))

    return max_score, aligned_seq1, aligned_seq2


# --- Execution Example ---
if __name__ == "__main__":
    # Test sequences containing partially shared local motifs
    sequence_a = "HEAGAWGHEE"
    sequence_b = "PAWHEAE"

    score, align1, align2 = smith_waterman(sequence_a, sequence_b, match=3, mismatch=-3, gap=-2)

    print(f"Optimal Local Alignment Score: {score}")
    print(f"Aligned Sequence 1: {align1}")
    print(f"Aligned Sequence 2: {align2}")


Optimal Local Alignment Score: 11
Aligned Sequence 1: AWGHE-E
Aligned Sequence 2: AW-HEAE
